# Step 8: Final Open-Vocabulary Anomaly Evaluation (RbA)

This notebook implements the final stage of the project: evaluating the **Efficient Open-Vocabulary Multi-Task (EoMT)** model on anomaly segmentation benchmarks using the **Rejected by All (RbA)** methodology.

### Objectives:
1. **Reconstruct Dense Maps:** Map query-based mask predictions to dense spatial logit maps.
2. **Implement RbA Score:** Apply the $\tanh$-based scoring function to identify out-of-distribution (OOD) pixels.
3. **Temperature Scaling:** Optimize the scoring threshold for better calibration.
4. **Cross-Checkpoint Evaluation:** Generate the final results table for the project report.

## 1. Environment & Imports
We treat the `/eomt` directory as a library. Ensure the project root is in your Python path.

In [1]:
# @title
!pip install ood_metrics > /dev/null ## It will restart the session

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.

In [1]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null
!pip install -U "torchao>=0.16.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
!pip install peft

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ln -s /content/drive/MyDrive/FundGitHubProject/ /content/Fundamental_Project # symbolic shortcut

In [4]:
%cd Fundamental_Project/

/content/drive/MyDrive/FundGitHubProject


In [5]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add project root and eomt directory to path
project_root = '/content/drive/MyDrive/FundGitHubProject/'
eomt_path = os.path.join(project_root, 'eomt')

# Add both directories to Python's module search path (sys.path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_path not in sys.path:
    sys.path.insert(0, eomt_path)

print("Contents of project_root:", os.listdir(project_root) if os.path.exists(project_root) else "Not found")
print("Contents of eomt_path:", os.listdir(eomt_path) if os.path.exists(eomt_path) else "Not found")

from eomt.models.eomt import EoMT
from eval.iouEval import iouEval

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Contents of project_root: ['eval', '.git', 'LICENSE', 'coco-classes-mapping-master', 'docs', 'gitfunctions', 'eomt', 'trained_models', 'README_AI_Guide.md', '.ipynb_checkpoints', 'README.md', 'wandb', 'inference.ipynb', 'lightning_logs', 'checkpoints', 'RbA-main', 'baselines', 'temperature_scaling-master', '.gitignore', 'requirements.txt', 'results.txt', 'Step6.ipynb', 'Step7.ipynb', 'Step5.ipynb', 'Step4.ipynb', 'finetuning_results.json', 'Step5_Enhanced.ipynb', 'FinalStep.ipynb']
Contents of eomt_path: ['.gitignore', 'LICENSE', '__init__.py', 'README.md', 'inference.ipynb', 'requirements.txt', 'main.py', 'configs', 'docs', 'training', 'datasets', 'models', 'data', '__pycache__', 'eomt_weights', '.ipynb_checkpoints', 'run_cli_silenced.py', 'wandb', 'checkpoint_utils.py', 'README_AI_Guide.md']
Using device: cuda


## 2. The RbA Scoring Engine
Unlike standard models, EoMT uses query-based mask classification. We must bridge the gap by reconstructing a dense map before scoring.

**Formula:**
$$L_k(x) = \sum_{n=1}^{N} P_{nk} M_n(x)$$
$$RbA(x) = -\sum_{k=1}^{K} \tanh(L_k(x))$$

In [6]:
def get_rba_anomaly_map(mask_cls, mask_pred, temperature=1.0):
    """
    Args:
        mask_cls: [B, N, C+1] raw class logits
        mask_pred: [B, N, H, W] raw mask logits
        temperature: Scaling factor for calibration
    """
    # 1. Apply Temperature and Softmax to get probabilities
    # Drop the last 'no-object' class index
    class_probs = F.softmax(mask_cls / temperature, dim=-1)[..., :-1] # Pay attention if the class are 19

    # 2. Sigmoid for mask probabilities
    mask_probs = torch.sigmoid(mask_pred)

    # 3. Dense Reconstruction via Einstein Summation
    # (Batch, Query, Class) x (Batch, Query, H, W) -> (Batch, Class, H, W)
    pixel_probs = torch.einsum('bnc,bnhw->bchw', class_probs, mask_probs)

    # 4. RbA Scoring (tanh variant)
    # Anomaly score is higher when all classes 'reject' the pixel
    anomaly_map = -torch.sum(torch.tanh(pixel_probs), dim=1)

    return anomaly_map

## 3. Model Loading from checkpoint
We load the weights into the EoMT architecture. Ensure you have the correct configuration parameters used during training.

In [7]:
# We can import directly from 'models' since the 'eomt' directory is in sys.path
# from models.eomt import EoMT # This import is redundant and may cause issues.
from eval.iouEval import iouEval


In [9]:
import torch
from eomt.checkpoint_utils import get_finetuned_model

ckpt_path = '/content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt'

eomt_ft = get_finetuned_model(ckpt_path)


--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights from: /content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.


In [12]:
import torch

# Load the raw checkpoint
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

# PyTorch Lightning usually stores weights in 'state_dict'
state_dict = ckpt.get('state_dict', ckpt)

# Strip 'model.' or 'net.' prefixes if they exist (common in Lightning)
cleaned_state_dict = {}
for k, v in state_dict.items():
    if k.startswith('model.'):
        cleaned_state_dict[k.replace('model.', '', 1)] = v
    elif k.startswith('net.'):
        cleaned_state_dict[k.replace('net.', '', 1)] = v
    else:
        cleaned_state_dict[k] = v

# Load into our model with strict=False to catch the mismatches
incompatible_keys = eomt_ft.load_state_dict(cleaned_state_dict, strict=False)

print("--- Key Mismatch Report ---")
print(f"Number of Unexpected Keys: {len(incompatible_keys.unexpected_keys)}")
if incompatible_keys.unexpected_keys:
    print("Unexpected Keys:")
    for k in incompatible_keys.unexpected_keys:
        print(f"  - {k}")

print(f"\nNumber of Missing Keys: {len(incompatible_keys.missing_keys)}")
if incompatible_keys.missing_keys:
    print("Missing Keys (showing first 5):")
    for k in incompatible_keys.missing_keys[:5]:
        print(f"  - {k}")

--- Key Mismatch Report ---
Number of Unexpected Keys: 270
Unexpected Keys:
  - network.attn_mask_probs
  - network.encoder.base_model.model.pixel_mean
  - network.encoder.base_model.model.pixel_std
  - network.encoder.base_model.model.backbone.cls_token
  - network.encoder.base_model.model.backbone.reg_token
  - network.encoder.base_model.model.backbone.pos_embed
  - network.encoder.base_model.model.backbone.patch_embed.proj.weight
  - network.encoder.base_model.model.backbone.patch_embed.proj.bias
  - network.encoder.base_model.model.backbone.blocks.0.norm1.weight
  - network.encoder.base_model.model.backbone.blocks.0.norm1.bias
  - network.encoder.base_model.model.backbone.blocks.0.attn.qkv.base_layer.weight
  - network.encoder.base_model.model.backbone.blocks.0.attn.qkv.base_layer.bias
  - network.encoder.base_model.model.backbone.blocks.0.attn.qkv.lora_A.default.weight
  - network.encoder.base_model.model.backbone.blocks.0.attn.qkv.lora_B.default.weight
  - network.encoder.base_mo

## Evaluation

In [ ]:
def run_full_evaluation(model, dataloader, temp=1.0):
    """
    Runs inference and calculates AUPRC and FPR@95
    """
    from sklearn.metrics import average_precision_score
    from ood_metrics import fpr_at_95_tpr

    model.eval()
    all_anomaly_scores = []
    all_gt_masks = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Anomaly Datasets"):
            images = batch['image'].to(device)
            gt = batch['label'] # Anomaly ground truth

            # Forward Pass
            mask_logits_list, class_logits_list = model(images)

            # Use final layer outputs
            m_pred = mask_logits_list[-1]
            c_cls = class_logits_list[-1]

            # Compute Score Map
            anomaly_map = get_rba_anomaly_map(c_cls, m_pred, temperature=temp)

            # Collect for metrics (ensure CPU and flattened)
            all_anomaly_scores.append(anomaly_map.cpu().numpy().flatten())
            all_gt_masks.append(gt.numpy().flatten())

    flat_scores = np.concatenate(all_anomaly_scores)
    flat_gt = np.concatenate(all_gt_masks)

    # Filter out ignore labels (usually 255)
    valid_mask = flat_gt != 255
    val_out = flat_scores[valid_mask]
    val_label = flat_gt[valid_mask]

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    return {'auprc': prc_auc * 100.0, 'fpr95': fpr * 100.0}

## 5. Final Results Table
Execute the evaluation across the three main checkpoints to complete your report.

In [ ]:



checkpoints = [
    "/content/Fundamental_Project/eomt/eomt_weights/eomt_cityscapes_lightning.ckpt",
    "/content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt"
]







print("| Checkpoint | Dataset | AUPRC | FPR@95 |")
print("|------------|---------|-------|--------|")
for ckpt in checkpoints:
  # You might need to pass specific kwargs depending on your EoMT config
  model = load_trained_model(ckpt)
  for ds in datasets:
          res = run_full_evaluation(model, ds_loader)
          print(f"| {ckpt.split('/')[-1]} | {ds} | {res['auprc']:.2f} | {res['fpr95']:.2f} |")
